<a href="https://colab.research.google.com/github/aranagnost/drone-audio-classification/blob/main/colab_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Drone Audio Classification — Colab Training

**Before running:** Runtime > Change runtime type > T4 GPU

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the repo

In [ ]:
import os
import sys

REPO_DIR = '/content/drone-audio-classification'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/aranagnost/drone-audio-classification.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

## 3. Verify GPU and dataset

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

DATASET_ROOT = '/content/drive/MyDrive/drone-audio/datasets/Drone_Audio_Dataset/audio'
print('\nDataset root exists:', os.path.exists(DATASET_ROOT))
print('Subdirs:', os.listdir(DATASET_ROOT))

## 4. Create artifacts directory

In [ ]:
os.makedirs('artifacts/checkpoints', exist_ok=True)

## 5. Load the dataset

In [ ]:
import shutil
print("Copying dataset to local disk...")
shutil.copytree(
    '/content/drive/MyDrive/drone-audio/datasets/',
    '/content/datasets/'
)
print("Done.")

## 6. Train Stage 1 (binary: drone vs no-drone)

In [ ]:
MODEL = 'small_cnn_v2'  # options: small_cnn_v1, small_cnn_v2, small_cnn_v3, big_cnn_v1, 3_stages_cnn_v1

!PYTHONPATH={REPO_DIR} python training/train_stage1.py \
    --model {MODEL} \
    --dataset_root {DATASET_ROOT} \
    --num_workers 2

## 7. Train Stage 2 (motor count: 2/4/6/8 motors)

In [ ]:
!PYTHONPATH={REPO_DIR} python training/train_stage2.py \
    --model {MODEL} \
    --dataset_root {DATASET_ROOT} \
    --num_workers 2

## 8. Evaluate Stage 1 + Stage 2

In [ ]:
!PYTHONPATH={REPO_DIR} python training/eval.py \
    --model {MODEL} \
    --dataset_root {DATASET_ROOT}

## 9. Copy checkpoints to Drive for persistence

Colab resets between sessions — save checkpoints to Drive so you don't lose them.

In [ ]:
import shutil

CKPT_DEST = f'/content/drive/MyDrive/drone-audio/checkpoints'
os.makedirs(CKPT_DEST, exist_ok=True)
shutil.copytree('artifacts/checkpoints', CKPT_DEST, dirs_exist_ok=True)
print('Checkpoints saved to:', CKPT_DEST)